In [1]:
using LatticeDiracOperators
using LatticeQCD
using Gaugefields
using Wilsonloop
using LMP

[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
[ Info: Skipping precompilation since __precompile__(false). Importing LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018].
┌ Warning: Error requiring `Plots` from `LatticeQCD`
│   exception =
│    LoadError: ArgumentError: Package LatticeQCD does not have Plots in its dependencies:
│    - You may have a partially installed environment. Try `Pkg.instantiate()`
│      to ensure all packages in the environment are installed.
│    - Or, if you have LatticeQCD checked out for development and have
│      added Plots as a dependency but haven't updated your primary
│      environment's manifest file, try `Pkg.resolve()`.
│    - Otherwise you may need to report an issue with LatticeQCD
│    Stacktrace:
│      [1] macro expansion
│        @ .\loading.jl:1839 [inlined]
│      [2] macro expansion
│        @ .\lock.jl:267 [inlined]
│      [3] __require(into::Module, mod::Symbol)
│        @ Base .\loading.jl:1816
│      [4] #invoke_in_world#3


In [2]:
NX = 4
NY = 4
NZ = 4
NT = 4
Nwing = 0
Dim = 4
NC = 3
U = Initialize_4DGaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "hot")
x = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(x)
t = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(t)
x=x/sqrt(norm2(x))
t=t/sqrt(norm2(t))
gamma=Clifford()

16-element Vector{Spin_matrix}:
 [1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im 1.0 + 0.0im]
 [0.0 + 0.0im 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im 1.0 + 0.0im; 1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im]
 [0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im 0.0 - 1.0im; 0.0 + 0.0im 0.0 + 0.0im 0.0 + 1.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 1.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 - 1.0im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im]
 [0.0 + 0.0im 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im -1.0 + 0.0im; -1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im]
 [0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im -1.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im 

In [3]:
oW=Random_SpinM("real")
W=Clifford(oW)

4×4 Spin_matrix:
 -1.38475+0.0im       1.76984+1.47732im  …   1.55404-1.65688im
  1.76984-0.77006im  -2.46407+0.0im         -3.53792-1.50975im
  1.31183+1.50975im   2.43291+1.55148im     0.638116-0.77006im
 -1.55404-1.55148im  0.323917-1.50975im      1.78004+0.0im

In [4]:
y=norm2(W*(U[1]*x))
delta=1e-9


-0.017804392005864592 - 0.01576214854503474im-0.02651089656656854 + 0.0015508735748399373im0.008997396654338942 - 0.0005980679294564023im-0.015610914177378869 - 0.017521670209648635im


1.0e-9

In [5]:
delta = 1e-8  # 부동소수점 오차를 방지하기 위한 delta 값 조정

# Gradient 저장할 행렬
grad1 = similar(oW)
L1 = similar(oW)

# f 계산 (원래 값)
y_ij = similar(x)
for i in 1:4
    for j in 1:4
        y_ij += oW[i, j] * gamma[4 * (i-1) + j] *(U[1]* x)
    end
end
f = norm2(y_ij)

# 수치 미분을 통한 gradient 계산
for i in 1:4
    for j in 1:4
        # W perturbation 적용
        dW_plus = deepcopy(oW)
        dW_minus = deepcopy(oW)
        
        dW_plus[i, j] += delta
        dW_minus[i, j] -= delta

        # Perturbation을 적용한 y_ij 계산
        y_ij_plus = similar(x)
        y_ij_minus = similar(x)

        for a in 1:4
            for b in 1:4
                y_ij_plus += dW_plus[a, b] * gamma[4 * (a-1) + b] * (U[1]*x)
                y_ij_minus += dW_minus[a, b] * gamma[4 * (a-1) + b] *(U[1]* x)
            end
        end
        
        f_plus = norm2(y_ij_plus)
        f_minus = norm2(y_ij_minus)

        # 중앙 차분 방식으로 수치 미분
        grad1[i, j] = (f_plus - f_minus) / (2 * delta)
    end
end

# 해석적 미분 계산
for i in 1:4
    for j in 1:4
        L1[i, j] = inner_product(y_ij, gamma[4 * (i-1) + j] *(U[1]* x))
    end
end

# Hermitian 대칭성 적용
L1 = conj(L1) + L1

# 수치 미분과 해석적 미분 비교
println("Gradient Difference (L - grad): ", L1 - grad1)


-0.017804392005864592 - 0.01576214854503474im-0.02651089656656854 + 0.0015508735748399373im0.008997396654338942 - 0.0005980679294564023im-0.015610914177378869 - 0.017521670209648635im
-0.017804392005864592 - 0.01576214854503474im-0.02651089656656854 + 0.0015508735748399373im0.008997396654338942 - 0.0005980679294564023im-0.015610914177378869 - 0.017521670209648635im
-0.017804392005864592 - 0.01576214854503474im-0.02651089656656854 + 0.0015508735748399373im0.008997396654338942 - 0.0005980679294564023im-0.015610914177378869 - 0.017521670209648635im
-0.017804392005864592 - 0.01576214854503474im-0.02651089656656854 + 0.0015508735748399373im0.008997396654338942 - 0.0005980679294564023im-0.015610914177378869 - 0.017521670209648635im
-0.017804392005864592 - 0.01576214854503474im-0.02651089656656854 + 0.0015508735748399373im0.008997396654338942 - 0.0005980679294564023im-0.015610914177378869 - 0.017521670209648635im
-0.017804392005864592 - 0.01576214854503474im-0.02651089656656854 + 0.0015508735

In [6]:
println("Gradient Difference (L - grad): ", L1 - grad1)

Gradient Difference (L - grad): ComplexF64[8.921454449639654e-7 + 0.0im -1.9860647593183955e-6 + 0.0im -2.0996166534281713e-6 + 0.0im 1.6823064510340657e-6 + 0.0im; 2.6980420273492456e-6 + 0.0im 8.98117989756031e-7 + 0.0im 4.278675813651489e-8 + 0.0im 1.636333461618733e-6 + 0.0im; -2.3601359124825194e-6 + 0.0im 2.407437792451006e-6 + 0.0im -1.1642119179811061e-6 + 0.0im 1.0980848674790877e-7 + 0.0im; 2.145351029492204e-6 + 0.0im -3.0424720248375947e-6 + 0.0im -3.059772039737396e-6 + 0.0im 3.758579319335098e-6 + 0.0im]
